# Welcome!
This notebook demonstrates how to develop a conversational system that uses a deep knowledge base about hotels, combining structured instance-level data and an ontological model. The knowledge graph (KG) and ontology enable reasoning to enhance dialogue response generation. The task involves integrating a GraphRAG-like approach to query the knowledge base and generate accurate, context-aware responses.

Specifically, the notebook has the following steps:

1. **Setup**: Loading the knowledge graph, dialogues, and required libraries (e.g., OWLAPY).
2. **Analyzing the knowledge graph**: Exploring its structure and entities using OWLAPY queries.
3. **Extending the ontology**: Adding TBox information for expressive reasoning.
4. **Creating dialogues**: Create dialogues based on the examples. Write 5 simple dialogues and 5 more detailed ones to showcase different types of interactions.
5. **Combining ontology and KG data**: Deploying an OWL reasoner to perform class-expression queries.
6. **Query generation with LLMs**: Using an LLM (e.g., Llama3.2) to generate or assist in creating queries against the KG.
7. **Generating responses**: Summarizing retrieved data into dialogue responses using a KG-augmented RAG approach.
8. **Evaluation**: Assessing the system's performance using metrics like intersection-over-union scores.

## Assignment
The goal of this assignment is to develop a logic-enhanced conversational system that retrieves and reasons over domain knowledge to assist in dialogue response generation. You will focus on both the technical aspects of KG+ontology reasoning and the integration with LLMs for robust responses.

### Assignment Steps
1. **Analyze the provided knowledge graph and dialogues**:
   - Explore the KG's entities, properties, and relevance to the dialogues.
   - Identify opportunities where ontology reasoning enhances dialogue responses.
2. **Extend the ontology**:
   - Add expressive TBox information to support meaningful inferences.
3. **Deploy the reasoning environment**:
   - Use OWLAPY to combine the KG (as ABox) with the ontology for reasoning-based queries.
4. **Generate class-expression queries**:
   - Use instruction-based, few-shot prompting with Llama3.2 to produce or assist in creating the queries.
5. **Summarize results into dialogue responses**:
   - Apply KG-augmented RAG to generate user-facing answers based on reasoning results.
6. **Evaluate the system**:
   - Use appropriate metrics, including intersection-over-union scores for set-based answers.

## Report
Write a **5-page report** in LNCS format that includes:

1. **Introduction**: Background on conversational systems with LLMs and the role of reasoning over domain knowledge.
2. **Methodology**: A detailed description of your approach, including diagrams and examples.
3. **Results**: Evaluation findings from the implemented steps.
4. **Discussion**: Strengths and weaknesses of your approach, lessons learned, and potential improvements.

Make sure to use the following template: [Springer Lecture Notes in Computer Science](https://www.overleaf.com/latex/templates/springer-lecture-notes-in-computer-science/kzwwpvhwnvfj)


## Grading
Your work will be evaluated based on:

1. **Code Implementation (30%)**: Quality and functionality of the logic-enhanced conversational system.
2. **Report (70%)**: Depth of analysis and clarity in presenting methods, results, and lessons learned.

## Kaggle Environment Notes
To ensure smooth execution:
- Load the required data into `/kaggle/input/`.
- Use `/kaggle/working/` for saving temporary files.
- Turn on GPUs and internet connectivity when necessary, and follow best practices for resource management.

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input director

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/w1-dataset3/extended_data.ttl


# Install packages

In [3]:
!pip install jpype1==1.5.2
!pip install owlapy==1.5.1
!pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.5/493.5 kB 8.7 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 75.2 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 31.8 MB/s eta 0:00:00
  Created wheel for owlready2: filename=owlready2-0.49-py3-none-any.whl size=23742426 sha256=e256cf7fc1d631d0bde3dd6ea3492a815d3467ba5caf68cb8f6cf8f9673f269a
  Stored in directory: /root/.cache/pip/wheels/43/fe/dc/a0de3c289cfd5923ece6524469d328950e14fa0c90b1088ffa
Successfully built owlready2


# Import libraries


In [4]:
from owlapy import manchester_to_owl_expression, dl_to_owl_expression
from owlapy.iri import IRI
from owlapy.owl_ontology import Ontology
from owlapy.owl_reasoner import SyncReasoner, StructuralReasoner

# 1. Analyze the provided knowledge graph (data.ttl).

In [5]:
## the provided knowledge graph is in turtle (.ttl) format, which owlready2 has trouble
## parsing correctly in this environment. to avoid this issue, we first load the file
## using rdflib (which fully supports turtle), convert it to n-triples,
## and then load the converted graph into owlready2 for analysis.
## for the record owlready2 is an inner library used by owlapy.

from rdflib import Graph
from owlready2 import World

src = "/kaggle/input/w1-dataset3/extended_data.ttl"
dst = "/kaggle/working/data.nt"   ## .nt is the file format for n-triples

g = Graph()

g.parse(src, format="turtle")   ## here we parse the .ttl
g.serialize(destination=dst, format="nt")   # here its converted into .nt

world = World()
onto = world.get_ontology(f"file://{dst}").load(format="ntriples")

print("loaded into:", onto.base_iri)

loaded into: file:///kaggle/working/data.nt#


/usr/local/lib/python3.12/dist-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [6]:
from collections import Counter
from rdflib import URIRef, Literal
import pandas as pd

# helper funcs
def is_uri(x): 
    return isinstance(x, URIRef)

def is_lit(x):
    return isinstance(x, Literal)

def shorten(term, graph):
    # compact display using namespaces when possible
    
    if isinstance(term, URIRef):
        try:
            return term.n3(graph.namespace_manager)
        except Exception:
            return str(term)
    if isinstance(term, Literal):
        if term.language:
            return f"\"{str(term)[:60]}\"@{term.language}"
        if term.datatype:
            return f"\"{str(term)[:60]}\"^^{term.datatype}"
        return f"\"{str(term)[:60]}\""
    return str(term)


triples = list(g.triples((None, None, None)))
print("--- basic info ---")
print("triples:", len(triples))

subjects = set(s for s, p, o in triples)
predicates = set(p for s, p, o in triples)
objects = set(o for s, p, o in triples)

uris = set(x for x in subjects.union(objects) if is_uri(x))
lits = set(x for x in objects if is_lit(x))

print("unique subjects:", len(subjects))
print("unique predicates:", len(predicates))
print("unique objects:", len(objects))
print("unique URI nodes (subjects U objects):", len(uris))
print("unique literal nodes (objects):", len(lits))

pred_counts = Counter(p for s, p, o in triples)
top_preds = pred_counts.most_common(30)

print()
print("--- top predicates (by triple count) ---")

df_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "count"]
)
display(df_preds.head(30))

RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

type_triples = list(g.triples((None, RDF_TYPE, None)))
class_counts = Counter(o for s, p, o in type_triples if is_uri(o))

print()
print("--- types / classes (rdf:type) ---")
print("rdf:type triples:", len(type_triples))
print("distinct classes:", len(class_counts))

df_classes = pd.DataFrame(
    [(str(cls), shorten(cls, g), c) for cls, c in class_counts.most_common()],
    columns=["class_iri", "class", "instances_count"]
)
display(df_classes.head(30))

datatype_counts = Counter()
lang_counts = Counter()
lit_pred_counts = Counter()
lit_lengths = []

for s, p, o in triples:
    if is_lit(o):
        lit_pred_counts[p] += 1
        if o.datatype:
            datatype_counts[o.datatype] += 1
        else:
            datatype_counts[None] += 1
        if o.language:
            lang_counts[o.language] += 1
        lit_lengths.append(len(str(o)))

print()
print("--- literals ---")
print("literal objects:", sum(lit_pred_counts.values()))
print("predicates with literals:", len(lit_pred_counts))
print("avg literal length:", (sum(lit_lengths) / len(lit_lengths)) if lit_lengths else 0)

print("top literal predicates:")
for p, c in lit_pred_counts.most_common(20):
    print(f"{c:>7}  {shorten(p, g)}")

print("top datatypes:")
for dt, c in datatype_counts.most_common(15):
    dt_name = "no-datatype" if dt is None else shorten(dt, g)
    print(f"{c:>7}  {dt_name}")

print("top languages:")
for lang, c in lang_counts.most_common(10):
    print(f"{c:>7}  {lang}")

df_lit_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in lit_pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "literal_count"]
)
display(df_lit_preds.head(30))

--- basic info ---
triples: 9237
unique subjects: 1789
unique predicates: 14
unique objects: 870
unique URI nodes (subjects U objects): 1792
unique literal nodes (objects): 476

--- top predicates (by triple count) ---


,predicate_iri,predicate,count
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,rdf:type,3751
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:hasFacility,1640
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:location,1085
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:userRating,1000
4,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nationality,472
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:restaurantType,305
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:inCity,267
8,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:diet,137
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nextTo,34



--- types / classes (rdf:type) ---
rdf:type triples: 3751
distinct classes: 32


,class_iri,class,instances_count
0,http://www.w3.org/2002/07/owl#NamedIndividual,owl:NamedIndividual,1746
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Restaurant,506
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hotel,344
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Camping_Site,342
4,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hostel,314
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Neighbourhood,267
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Museum,55
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:City,34
8,http://www.w3.org/2002/07/owl#Class,owl:Class,33
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Trainstation,22



--- literals ---
literal objects: 485
predicates with literals: 1
avg literal length: 13.393814432989691
top literal predicates:
    485  rdfs:label
top datatypes:
    485  no-datatype
top languages:


,predicate_iri,predicate,literal_count
0,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485


# 2. Create a small ontology that can support expressive inference about hotels and analyse the dialogues (examples.txt).

# Create your own dialogues

Once you have created your ontology, use it as the foundation for designing dialogues. Study the examples in examples.txt to understand their structure and content. Then, create 10 dialogues of your own, ensuring a range of difficulty levels: 5 simple ones and 5 more challenging ones. These dialogues should illustrate how your ontology can support reasoning and should include references to the types of information modeled in your ontology.

In [ ]:
# Create 10 dialoges based on the description
dialogue1: str = ""
dialogue2: str = ""
dialogue3: str = ""
dialogue4: str = ""
dialogue5: str = ""
dialogue6: str = ""
dialogue7: str = ""
dialogue8: str = ""
dialogue9: str = ""
dialogue10: str = ""
dialogues: list = [dialogue1, dialogue2, dialogue3, dialogue4, dialogue5, dialogue6, dialogue7, dialogue8, dialogue9, dialogue10]

# 3. Deploy a reasoning environment

Treated as the ABox in the OWL knowledge base. The idea is that instance queries
with complex class expressions should be used to retrieve different hotels, where
reasoning is crucial for many aspects. For example, a query "give me places that are
close to a coast" would also return places next to a beach if the query is evaluated
together with the ontology that states that a place next to the beach is next to a
coast.

In [7]:
ontology_path: str = dst

In [8]:
from owlready2 import World, sync_reasoner_pellet, ThingClass

world = World()
kb = world.get_ontology(f"file://{ontology_path}").load(format="ntriples")
with kb:
    sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)

print("kb loaded + reasoned:", kb.base_iri)


# helpers
def _find_class_by_name(name: str):
    name_l = (name or "").lower()
    for c in kb.classes():
        if (c.name or "").lower() == name_l:
            return c
    return None

def _find_objprop_by_name(name: str):
    name_l = (name or "").lower()
    for p in kb.object_properties():
        if (p.name or "").lower() == name_l:
            return p
    return None

def _find_individual_by_name_or_label(name: str):
    """
    Accepts:
      - ind.name (e.g., "Q142")
      - rdfs:label (e.g., "France")
      - full IRI (e.g., "http://www.wikidata.org/entity/Q142")
    """
    if not name:
        return None

    s = str(name).strip()

    # 1) If a full IRI is given, try direct lookup
    if s.startswith("http://") or s.startswith("https://"):
        ent = kb.world[s]  # Owlready2 direct IRI lookup
        # return only if it's an individual (NamedIndividual-ish)
        if ent is not None and not isinstance(ent, ThingClass):
            return ent

    target = s.lower()

    # 2) Match by .name or rdfs:label
    for ind in kb.individuals():
        if (ind.name or "").lower() == target:
            return ind
        try:
            labels = list(getattr(ind, "label", []))
        except Exception:
            labels = []
        for lab in labels:
            if (str(lab) or "").lower() == target:
                return ind

    return None

def _suggest(names, target, k=15):
    target = (target or "").lower()
    hits = [n for n in names if target in (n or "").lower()]
    return hits[:k] if hits else names[:k]

def query_to_nl(node: dict) -> str:
    ## recommended by the TA
    """
    Examples:
      {"type":"class","class":"Hotel"}  -> "Hotel"
      {"type":"exists","property":"hasFacility","filler":"Facility"} -> "hasFacility some Facility"
      {"type":"value","property":"inCountry","individual":"Italy"}   -> "inCountry value Italy"
    """
    if not isinstance(node, dict):
        return "<?>"

    t = node.get("type")

    # base cases
    if t == "class":
        return node.get("class", "?Class")

    if t == "value":
        p = node.get("property", "?prop")
        i = node.get("individual", "?Individual")
        return f"{p} value {i}"

    if t == "exists":
        p = node.get("property", "?prop")
        filler = node.get("filler", "?")
        if isinstance(filler, dict):
            return f"{p} some ({query_to_nl(filler)})"
        return f"{p} some {filler}"

    # boolean cases
    if t in ("and", "or"):
        op = " AND " if t == "and" else " OR "
        parts = [query_to_nl(x) for x in node.get("operands", [])]
        if not parts:
            return f"{t.upper()}(?)"
        # parenthesize any child that’s compound
        def par(s: str) -> str:
            return f"({s})" if (" AND " in s or " OR " in s or s.startswith("NOT(")) else s
        return op.join(par(s) for s in parts)

    if t == "not":
        return f"NOT({query_to_nl(node.get('operand', {}))})"

    # shorthand format: {"class":"Hotel","exists":[...]}  ->  Hotel AND (p some C) AND ...
    if t is None and ("class" in node or "exists" in node):
        parts = []
        if "class" in node:
            parts.append(node["class"])
        for ex in node.get("exists", []):
            filler = ex.get("filler", ex.get("filler_class", "?Class"))
            parts.append(f"{ex.get('property','?prop')} some {filler}")
        return " AND ".join(parts) if parts else "<?>"

    return "UNABLE TO TRANSLATE TO NATURAL LANGUAGE"


# retriever over reasoner
def reasoner_instances(query: dict, limit: int = 25):
    """
    NOW supports:
      - {"type":"value","property":"inCountry","individual":"Italy"}
      - exists.filler can be a nested dict expression (not only a class string)
    """

    available_classes  = [c.name for c in kb.classes()]
    available_objprops = [p.name for p in kb.object_properties()]
    available_inds     = [getattr(i, "name", None) for i in kb.individuals()]

    # build a normalized tree-form for evaluation
    def normalize(node):
        if not isinstance(node, dict):
            raise ValueError(f"node must be dict, got {type(node)}: {node}")

        t = node.get("type")

        # case class
        if t == "class":
            cname = node.get("class")
            C = _find_class_by_name(cname)
            if C is None:
                raise ValueError(f"class '{cname}' not found. suggestions: {_suggest(available_classes, cname)}")
            return ("class", C)

        # case value
        if t == "value":
            pname = node.get("property")
            iname = node.get("individual") or node.get("value")

            P = _find_objprop_by_name(pname)
            if P is None:
                raise ValueError(
                    f"object property '{pname}' not found. suggestions: {_suggest(available_objprops, pname)}"
                )

            I = _find_individual_by_name_or_label(iname)
            if I is None:
                # show some individual names as hint (labels may be too many)
                some = [n for n in available_inds if n][:25]
                raise ValueError(
                    f"individual '{iname}' not found by name or rdfs:label. "
                    f"Try 'Q38' or the exact label. Example individual names: {some}"
                )

            return ("value", P, I)

        # case exists (filler can be str class OR nested dict expr)
        if t == "exists":
            pname = node.get("property")
            filler = node.get("filler")

            P = _find_objprop_by_name(pname)
            if P is None:
                raise ValueError(
                    f"object property '{pname}' not found. suggestions: {_suggest(available_objprops, pname)}"
                )

            # filler is nested expression
            if isinstance(filler, dict):
                F_ast = normalize(filler)
                return ("exists", P, F_ast)

            # filler is class name (string)
            F = _find_class_by_name(filler)
            if F is None:
                raise ValueError(f"filler class '{filler}' not found. suggestions: {_suggest(available_classes, filler)}")
            return ("exists", P, ("class", F))

        # case and/or
        if t in ("and", "or"):
            ops = node.get("operands", [])
            if not ops:
                raise ValueError(f"'{t}' node has no operands")
            return (t, [normalize(x) for x in ops])

        # case negation
        if t == "not":
            op = node.get("operand")
            if op is None:
                raise ValueError("'not' node missing 'operand'")
            return ("not", normalize(op))

        # case implicit conjunction (shorthand query format)
        ## example: Hotel n EhasFacility.Facility instead of explicitly encoding
        ## as AND(("class", Hotel), ("exists", hasFacility, Facility))
        if t is None and ("class" in node or "exists" in node):
            ops = []
            if "class" in node:
                ops.append({"type": "class", "class": node["class"]})
            for ex in node.get("exists", []):
                # allow shorthand exists items to themselves have dict fillers too
                ops.append({"type": "exists", "property": ex["property"], "filler": ex.get("filler", ex.get("filler_class"))})
            if len(ops) == 1:
                return normalize(ops[0])
            return ("and", [normalize(x) for x in ops])

        raise ValueError(f"unknown query node format: {node}")

    ast = normalize(query)

    # membership checks on individuals (after pellet)
    def is_a_or_subclass(ind, C):
        # direct asserted/inferred types
        for t in ind.is_a:
            if t == C:
                return True
            # if both are OWL classes, allow subclass match
            if isinstance(t, ThingClass) and isinstance(C, ThingClass):
                try:
                    if issubclass(t, C):
                        return True
                except TypeError:
                    pass
        return False

    def eval_ast(ind, node):
        tag = node[0]

        if tag == "class":
            C = node[1]
            return is_a_or_subclass(ind, C)

        if tag == "value":
            P, I = node[1], node[2]
            vals = getattr(ind, P.name, [])
            return any(v == I for v in vals)

        if tag == "exists":
            P, filler_ast = node[1], node[2]
            vals = getattr(ind, P.name, [])
            return any(eval_ast(v, filler_ast) for v in vals)

        if tag == "and":
            return all(eval_ast(ind, child) for child in node[1])

        if tag == "or":
            return any(eval_ast(ind, child) for child in node[1])

        if tag == "not":
            return not eval_ast(ind, node[1])

        raise RuntimeError("unexpected AST node")

    # if query contains a base class, start from its instances
    def extract_base_class(node):
        if node[0] == "class":
            return node[1]
        if node[0] in ("and", "or"):
            for ch in node[1]:
                bc = extract_base_class(ch)
                if bc is not None:
                    return bc
        if node[0] == "not":
            return extract_base_class(node[1])
        return None

    base = extract_base_class(ast)
    candidates = list(base.instances()) if base is not None else list(kb.individuals())
    results = [ind for ind in candidates if eval_ast(ind, ast)]

    print("\nquery spec:", query)
    print("matches:", len(results))
    print("query (natural language):", query_to_nl(query))
    for ind in results[:limit]:
        print(" -", ind.iri)

    return results

# example queries
q1 = reasoner_instances({"type":"class", "class":"Hotel"})

q2 = reasoner_instances({
    "type": "exists",
    "property": "hasFacility",
    "filler": "Facility"
})

q3 = reasoner_instances({
    "type": "and",
    "operands": [
        {"type":"class", "class":"Hotel"},
        {"type":"exists", "property":"hasFacility", "filler":"Facility"}
    ]
})

q4 = reasoner_instances({
    "type": "or",
    "operands": [
        {"type":"class", "class":"Hotel"},
        {"type":"class", "class":"Hostel"}
    ]
})

q5 = reasoner_instances({
    "type": "and",
    "operands": [
        {"type":"class", "class":"Hotel"},
        {"type":"not", "operand": {"type":"class", "class":"Hostel"}}
    ]
})

q6 = reasoner_instances({"type":"class", "class":"Hostel"})

q7 = reasoner_instances({
  "type":"and",
  "operands":[
    {"type":"class","class":"Hotel"},
    {"type":"not","operand":{"type":"exists","property":"hasFacility","filler":"Facility"}}
  ]
})

# example query with specific instance value
q_italy = reasoner_instances({
    "type": "and",
    "operands": [
        {"type": "class", "class": "City"},
        {"type": "value", "property": "inCountry", "individual": "Italy"}
    ]
})

* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/pellet/httpcore-4.2.2.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jcl-over-slf4j-1.6.4.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jgrapht-jdk1.5.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/slf4j-log4j12-1.6.4.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/commons-codec-1.6.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/httpclient-4.2.3.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/log4j-core-2.19.0.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/antlr-runtime-3.2.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/antlr-3.2.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/slf4j-api-1.6.4.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jena-iri-0.9.5.jar:/usr/local/lib

kb loaded + reasoned: file:///kaggle/working/data.nt#

query spec: {'type': 'class', 'class': 'Hotel'}
matches: 344
query (natural language): Hotel
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation817
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation975
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation176
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation652
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation987
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation733
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation360
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation984
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation493
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation87
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation12
 - http://kai.cs.vu.nl/2024/situated-minor-project/

# 4. Instruct the LLM to produce the query or components of the query (e.g., keywords) against the KG

In [9]:
#Download ollama
# For Kaggle or Linux: download with this command, for Windows & Mac locally, download executable from website
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
process = subprocess.Popen("ollama serve", shell=True) #runs on a different thread

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tgz
#######################################################################   99.9%                                                   0.6%        18.7%                  63.8%############################################################ 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIEoQLwyHk+dcQ+zCzuKfANm0xUe1XOK5PqNr0rcahd5+



time=2026-01-11T10:21:16.974Z level=INFO source=routes.go:1554 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:4096 OLLAMA_DEBUG:INFO OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_REMOTES:[ollama.com] OLLAMA_SCHED_SP

In [10]:
# Import ollama & pull LLM
import ollama
!ollama pull llama3.2
model: str = "llama3.2"

[GIN] 2026/01/11 - 10:22:52 | 200 |      74.384µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ 

time=2026-01-11T10:22:53.063Z level=INFO source=download.go:177 msg="downloading dde5aa3fc5ff in 16 126 MB part(s)"


pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff:   2% ▕                  ▏  41 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   6% ▕█                 ▏ 117 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   8% ▕█                 ▏ 161 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  13% ▕██                ▏ 258 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  18% ▕███               ▏ 358 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  20% ▕███               ▏ 404 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  24% ▕████              ▏ 494 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  29% ▕█████             ▏ 592 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  32% ▕█████             ▏ 643 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:  37% ▕██████            ▏ 739 MB/2.0 GB          

time=2026-01-11T10:22:57.238Z level=INFO source=download.go:177 msg="downloading 966de95ca8a6 in 1 1.4 KB part(s)"


pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca

...

pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB              

time=2026-01-11T10:23:00.802Z level=INFO source=download.go:177 msg="downloading 56bb8bd477a5 in 1 96 B part(s)"


pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB              

time=2026-01-11T10:23:01.990Z level=INFO source=download.go:177 msg="downloading 34bb5ab01051 in 1 561 B part(s)"


pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         pullin

In [ ]:
#Step 1: Write the instruction for the LLM - remember the overarching topic (assistance with hotels), as well as the fact that
# this step is meant to merely extract queries from the user input.

# Instruct LLM
instruction: str = "..."

In [ ]:
#Step 2: Write a function that takes the model, instruction and one user question as input, runs the LLM and outputs its response
def question_to_query(instruction: str, question: str, model="llama3.2") -> str:
    '''
    This function is meant to use the instruction defined above to run the LLM in order to convert one user input
    question into a query for the ontology reasoner.
    Parameters: instruction (string), question (string), model version (string)
    Returns: LLM response (string)
    '''
    # TODO

In [ ]:
#Step 3: Run the LLM for each example defined above

# Helper function
def find_between(s: str, start: str, end: str) -> str:
    return s.split(start)[1].split(end)[0]

for dialogue in dialogues:
    print("User question:", dialogue)
    print()
    result: str = question_to_query(instruction, dialogue, model)
    # Possibly only extract the relevant parts
    print("Extracted query:", result)
    queries.append(result)
    print()

# 5. Use an LLM to summarize some result into a natural language response to the user.

In [26]:
#Step 1: Extract knowledge from query with the reasoner and return as list

import json

def reason(query: str) -> list:
    """
    query can be:
      - JSON string of the query dict
      - already a dict
    returns: list[str] of individual IRIs
    """
    if isinstance(query, str):
        query = query.strip()
        if query.startswith("{"):
            query_dict = json.loads(query)
        else:
            raise ValueError("JSON or dict required")
    elif isinstance(query, dict):
        query_dict = query
    else:
        raise TypeError("query must be a JSON string or dict")

    inds = reasoner_instances(query_dict, limit=25)
    return [str(ind.iri) for ind in inds]

In [27]:
#Step 2: Instruct & run the LLM for the new task: transform the extracted knowledge
# into a natural language response based on the original question

import textwrap

def _run_ollama(prompt: str, model: str = "llama3.2", timeout: int = 180, force_json: bool = False) -> str:
    """
    If force_json=True, TRIES to call ollama with JSON output mode.
    Not all ollama versions support this flag; if it fails, retries without it.
    """
    base_cmd = ["ollama", "run", model]

    if force_json:
        # if doesn't work, catch and retry without.
        cmd = base_cmd + ["--format", "json"]
    else:
        cmd = base_cmd

    try:
        p = subprocess.run(
            cmd,
            input=prompt,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"ollama run timed out after {timeout}s")

    # if JSON mode flag not supported
    if force_json and p.returncode != 0:
        err = (p.stderr or "") + (p.stdout or "")
        if "unknown flag" in err.lower() or "unknown shorthand flag" in err.lower():
            return _run_ollama(prompt, model=model, timeout=timeout, force_json=False)
            # retry

    if p.returncode != 0:
        err = (p.stderr or "").strip()
        if "ollama server not responding" in err.lower() or "could not connect" in err.lower():
            raise RuntimeError("Ollama server not running. Start it with: `ollama serve`")
        raise RuntimeError(
            "ollama run failed\n"
            f"returncode: {p.returncode}\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}\n"
        )

    return (p.stdout or "").strip()

def _entity_from_iri(iri: str):
    # owlready2 lets access to entities by IRI via world[iri]
    try:
        ent = world[iri]
        return ent
    except Exception:
        return None

def _best_label(ent) -> str | None:
    if ent is None:
        return None

    # rdfs:label is accessible via .label in owlready2
    try:
        if hasattr(ent, "label") and ent.label:
            # ent.label can be a list of strings
            return str(ent.label[0])
    except Exception:
        pass

    # fallback: local name or fragment
    try:
        if getattr(ent, "name", None):
            return str(ent.name)
    except Exception:
        pass

    # fallback: last part of IRI
    try:
        s = str(ent.iri) if hasattr(ent, "iri") else ""
        if "#" in s:
            return s.split("#", 1)[1]
        return s.rsplit("/", 1)[-1] if "/" in s else s
    except Exception:
        return None

def _safe_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]

def _summarize_individual(iri: str, max_facilities: int = 6) -> str:
    """
    Returns one readable line like:
      - HotelName (IRI) | facilities: Wifi, Pool | city: Rome | country: Italy
    
    If a property isn't in the ontology, it won't appear.
    """
    ent = _entity_from_iri(iri)
    if ent is None:
        return f"- {iri}"

    label = _best_label(ent) or "(no label)"
    parts = []

    # facilities (multiple)
    if hasattr(ent, "hasFacility"):
        facs = _safe_list(getattr(ent, "hasFacility", []))
        fac_labels = []
        for f in facs[:max_facilities]:
            fl = _best_label(f)
            if fl:
                fac_labels.append(fl)
        if fac_labels:
            parts.append("facilities: " + ", ".join(fac_labels))

    # other props
    for prop_name, pretty in [
        ("inCity", "City"),
        ("location", "Neighbourhood"),
        ("inCountry", "Country"),
        ("nationality", "Nationality"),
        ("nextTo", "Neighbourhood"),
        ("diet","Diet"),
        ("restaurantType", "RestaurantType"),
        ("userRating", "UserRating")
    ]:
        if hasattr(ent, prop_name):
            vals = _safe_list(getattr(ent, prop_name, []))
            val_labels = []
            for v in vals[:2]:
                vl = _best_label(v)
                if vl:
                    val_labels.append(vl)
            if val_labels:
                parts.append(f"{pretty}: " + ", ".join(val_labels))

    # if nothing found, at least return label + iri
    if parts:
        return f"- {label} | {iri} | " + " | ".join(parts)
    else:
        return f"- {label} | {iri}"


def enrich_knowledge_items(iris: list[str], k: int = 15) -> str:
    """
    Produces enriched bullet list text for the LLM prompt.
    """
    shown = iris[:k]
    hidden_n = max(0, len(iris) - k)

    lines = [ _summarize_individual(iri) for iri in shown ]
    if hidden_n:
        lines.append(f"... (+{hidden_n} more results)")
    return "\n".join(lines)


def knowledge_to_response(question: str, knowledge_items: list[str], model: str = "llama3.2", k: int = 15) -> str:
    """
    This function is meant to write an instruction based on an item of
    extracted knowledge and the original user question, and run the LLM to summarize
    a response.

    Note: Explicitly tells the LLM that the listed items ARE the matching hotels,
    and the enriched facts are passed rather than raw IRIs.
    """
    enriched = enrich_knowledge_items(knowledge_items, k=k)

    prompt = f"""You are answering a user question using ONLY the retrieved knowledge below.

    Important:
    - Each bullet is a MATCHING INDIVIDUAL from the knowledge graph (i.e., those are the hotels / entities that satisfy the query).
    - You may only use the facts shown in the bullets (labels / facilities / location).
    
    Question:
    {question}
    
    Retrieved matching individuals:
    {enriched if enriched.strip() else "(empty)"}
    
    Rules:
    - If the list is empty: say no matches found.
    - If bullets lack details needed to answer: say what detail is missing.
    - Otherwise answer in 1-3 sentences, concise and direct.
    """
    
    return _run_ollama(prompt, model=model)

In [29]:
#Step 3: Combine everything: generate queries from the dialogues,
#extract knowledge from queries with the reasoner and generate summary responses

def run_pipeline(question: str, query: str, model: str = "llama3.2", k: int = 15):
    print("Reasoner in progress...")
    print("Knowledge retrieved:")
    answers = reason(query)  # list[str]
    print("")
    response = knowledge_to_response(question=question, knowledge_items=answers, model=model, k=k)
    
    return {
        "question": question,
        "query": query,
        "knowledge": answers[:k],
        "response": response,
    }


result = run_pipeline("show me hotels that have a facility",
                      {'type': 'and',
                       'operands': [{'type': 'class',
                                     'class': 'Hotel'},
                                            {'type': 'exists',
                                            'property': 'hasFacility',
                                            'filler': 'Facility'}]})

print("Question:", result["question"], "\n")
print("Query:", result["query"], "\n")
print("Knowledge (first 15):")
for x in result["knowledge"]:
    print(" -", x)
print()
print("Final Response from oLlama:\n", result["response"])

Reasoner in progress...
Knowledge retrieved:

query spec: {'type': 'and', 'operands': [{'type': 'class', 'class': 'Hotel'}, {'type': 'exists', 'property': 'hasFacility', 'filler': 'Facility'}]}
matches: 324
query (natural language): Hotel AND hasFacility some Facility
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation817
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation975
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation176
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation652
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation987
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation733
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation360
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation493
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation87
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation12
 - 

# 6. Evaluate your LLM

In [ ]:
# TODO: your code to implement and demonstrate evaluation metrics
# Suggestions: comparison of generated queries with the queries manually created in examples.txt, Intersection Over Union,
# but you can be creative here